<div dir="rtl">

# المعمل 5: HuggingFace للرؤية الحاسوبية

في هذا المعمل، سنستكشف نظام HuggingFace لمهام الرؤية الحاسوبية. سنقوم بتغطية التضمينات المرئية (Visual Embeddings) باستخدام CLIP، والتفكير متعدد الوسائط (Multimodal Reasoning) باستخدام Qwen3.5-0.8B.

## 1. إعداد البيئة وإدارة الذاكرة

أولاً، نقوم بتثبيت المكتبات اللازمة وتعريف أداة مساعدة للحفاظ على نظافة ذاكرة وحدة معالجة الرسومات (GPU).

</div>


In [1]:
# تثبيت المكتبات ذات الصلة إذا كنت في Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q -U transformers accelerate bitsandbytes torch torchvision

import torch
import gc
from PIL import Image
import requests
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

def flush_memory():
    """أداة لمسح ذاكرة GPU بين التمارين لتجنب نفاذ الذاكرة (OOM)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("تم إفراغ الذاكرة.")

<div dir="rtl">

## 2. واجهة برمجة تطبيقات HuggingFace Pipeline

تعد واجهة `pipeline` أسرع طريقة لاستخدام النموذج. دعونا نصنف صورة لحافلة.

</div>


In [2]:
from transformers import pipeline

# التمرين 1: تصنيف الصور بدون تدريب مسبق (Zero-shot)
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
url = "https://ultralytics.com/images/bus.jpg"
result = classifier(url)

for r in result:
    print(f"{r['label']}: {r['score']:.4f}")


In [3]:
# تنظيف الذاكرة للنماذج التالية
del classifier
flush_memory()

<div dir="rtl">

## 2. التضمينات المرئية (CLIP)

تقوم التضمينات برسم خرائط للصور في مساحة دلالية. يربط CLIP النص بالصور.

</div>


In [4]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoImageProcessor


# التمرين 3: تصنيف CLIP بدون تدريب مسبق (Zero-shot)
clip_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_id).to("cuda" if torch.cuda.is_available() else "cpu")
clip_proc = CLIPProcessor.from_pretrained(clip_id)

image = Image.open(requests.get("https://ultralytics.com/images/bus.jpg", stream=True).raw)
labels = ["صورة لحافلة", "صورة لمشهد في المدينة", "صورة لصحراء"]
inputs = clip_proc(text=labels, images=image, return_tensors="pt", padding=True).to(clip_model.device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

for label, prob in zip(labels, probs[0]):
    print(f"{label}: {prob.item():.4f}")

del clip_model, clip_proc
flush_memory()

<div dir="rtl">

## 3. نماذج لغة الرؤية (Qwen3.5-0.8B)

نموذج Qwen3.5-0.8B خفيف الوزن هو نموذج لغة رؤية (VLM) ممتاز للاستدلال على الأجهزة ومهام الـ OCR.

</div>


In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-0.8B")
model = AutoModelForImageTextToText.from_pretrained("Qwen/Qwen3.5-0.8B")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://ultralytics.com/images/bus.jpg"},
            {"type": "text", "text": "صف ألوان المركبات في هذه الصورة."},
        ],
    }
]

inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


In [6]:
del model, processor
flush_memory()